# H-T05 → H-T08 SGD 학습률 튜닝

SGD를 고정하고 Tversky α/β와 학습률 조합 4개를 순차 실행합니다. 각 실험은 조기 종료 없이 300 epoch까지 학습합니다. 매 epoch checkpoint를 만들되, 5 epoch마다 Full-volume validation을 수행하고 Macro Dice 상위 3개 후보의 ±4 epoch만 보존합니다.


In [1]:
from pathlib import Path
import json, os, subprocess, time
import pandas as pd
from IPython.display import clear_output, display

ROOT = Path("/home/stu03/workspace/model")
DATA_ROOT = Path("/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet")
PYTHON = ROOT / ".venv/bin/python"
BATCH_DIR = ROOT / "runs/batches/H-T05_H-T08"
BATCH_STATUS = BATCH_DIR / "batch_status.json"
assert PYTHON.is_file() and DATA_ROOT.is_dir()


## 1. 설정·테스트·plans 확인

이 셀은 학습을 시작하지 않습니다. 설정과 보존 로직을 검사하고 네 개 custom plan을 생성합니다.


In [1]:
subprocess.run([str(PYTHON), "-m", "pytest", "-q", "tests/test_h_tuning_checkpoint_retention.py"], cwd=ROOT, check=True)
for name in ("H-T05", "H-T06", "H-T07", "H-T08"):
    config = f"configs/experiments/hemorrhage/tuning/{name}.yaml"
    subprocess.run([str(PYTHON), "-c", f"from brainon_ai.nnunet.config import load_config; load_config('{config}'); print('{name}: config OK')"], cwd=ROOT, check=True)
subprocess.run([str(PYTHON), "scripts/preprocessing/prepare_h_t05_h_t08_plans.py", "--preprocessed-root", str(DATA_ROOT / "nnUNet_preprocessed")], cwd=ROOT, check=True)
subprocess.run([str(PYTHON), "scripts/training/run_h_t05_h_t08.py", "--data-root", str(DATA_ROOT), "--dry-run"], cwd=ROOT, check=True)


NameError: name 'subprocess' is not defined

## 2. 네 실험 백그라운드 순차 실행

처음 실행은 `RESUME=False`입니다. 중단된 실험을 checkpoint에서 이어갈 때만 `True`로 변경하세요. 노트북 커널을 종료해도 학습은 계속됩니다.


In [7]:
RESUME = False
GPU = "0"
BATCH_DIR.mkdir(parents=True, exist_ok=True)
command = [str(PYTHON), "scripts/training/run_h_t05_h_t08.py", "--data-root", str(DATA_ROOT), "--gpu", GPU]
if RESUME:
    command.append("--resume")
launch_log = (BATCH_DIR / "launcher.log").open("a", encoding="utf-8")
process = subprocess.Popen(command, cwd=ROOT, stdout=launch_log, stderr=subprocess.STDOUT, start_new_session=True)
(BATCH_DIR / "launcher.pid").write_text(str(process.pid), encoding="utf-8")
print("PID:", process.pid)
print("log:", BATCH_DIR / "launcher.log")


PID: 326617
log: /home/stu03/workspace/model/runs/batches/H-T05_H-T08/launcher.log


## 3. 실시간 진행률

기존 학습 지표를 모두 유지하고 Full-volume Macro Dice, Top-3 후보와 현재 보존된 epoch checkpoint 수도 함께 표시합니다. Full validation 중에는 표 갱신이 잠시 늦어질 수 있습니다. 네 학습 후 Top-3 주변 정밀 검증까지 자동으로 이어집니다.


In [ ]:
PROGRESS_FILES = {
    "H-T05": ROOT / "runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTuneA04B06SgdLr005V1__3d_fullres/fold_0/progress.json",
    "H-T06": ROOT / "runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTuneA04B06SgdLr020V1__3d_fullres/fold_0/progress.json",
    "H-T07": ROOT / "runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTuneA02B08SgdLr005V1__3d_fullres/fold_0/progress.json",
    "H-T08": ROOT / "runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTuneA02B08SgdLr020V1__3d_fullres/fold_0/progress.json",
}

def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def fmt(seconds):
    if seconds is None:
        return "-"
    seconds = int(seconds); hours, remainder = divmod(seconds, 3600); minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

while True:
    clear_output(wait=True)
    batch = read_json(BATCH_STATUS); items = batch.get("experiments", {}); rows = []
    for name, path in PROGRESS_FILES.items():
        p = read_json(path); b = items.get(name, {})
        rows.append({
            "실험": name,
            "상태": p.get("status", b.get("status", "waiting")),
            "Epoch": f"{p.get('current_epoch', 0)}/{p.get('total_epochs', '-')}",
            "진행률(%)": p.get("progress_percent", 0),
            "ETA": fmt(p.get("eta_seconds")),
            "Loss": p.get("train_loss"),
            "Dice": p.get("pseudo_dice"),
            "Recall": p.get("pseudo_recall"),
            "Best Composite": p.get("best_composite"),
            "GPU 메모리(GB)": p.get("gpu_max_memory_gb"),
            "현재 Full Macro Dice": p.get("full_macro_dice"),
            "Best Full Macro Dice": p.get("best_full_macro_dice"),
            "Best Macro Epoch": p.get("best_full_macro_epoch"),
            "Top-3 Macro Epochs": p.get("top_full_macro_epochs", []),
            "보존 Checkpoints": p.get("retained_epoch_checkpoints", 0),
        })
    actual = [row["상태"] for row in rows]
    if batch.get("status") == "running":
        overall = "running"
    elif all(x in {"completed", "early_stopped"} for x in actual):
        overall = "completed"
    else:
        overall = batch.get("status", "starting")
    current = next((row["실험"] for row in rows if row["상태"] == "running"), batch.get("current", "-"))
    print("전체:", overall, "현재:", current or "-")
    display(pd.DataFrame(rows))
    if overall in {"completed", "failed"}:
        break
    time.sleep(10)


전체: running 현재: H-T05


,실험,상태,Epoch,진행률(%),ETA,Loss,Dice,Recall,Best Composite,GPU 메모리(GB),현재 Full Macro Dice,Best Full Macro Dice,Best Macro Epoch,Top-3 Macro Epochs,보존 Checkpoints
0,H-T05,running,66/300,22.0,02:37:44,-0.432503,0.788255,0.742332,0.831384,10.935,0.550379,0.567098,40.0,"[40, 45, 60]",25
1,H-T06,waiting,0/-,0.0,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
2,H-T07,waiting,0/-,0.0,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
3,H-T08,waiting,0/-,0.0,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0


## 4. Top-3 주변 정밀 검증 수동 재실행

각 실험의 5-epoch 단위 Macro Dice 상위 3개를 가져와 각 후보의 ±4 epoch를 전부 검증합니다. 겹치는 epoch는 한 번만 실행하고 최종 최고 checkpoint를 `checkpoint_best_full_macro.pth`로 저장합니다. 순차 실행 스크립트가 자동 수행합니다. 자동 검증이 중단됐을 때만 이 셀을 다시 실행하세요.


In [ ]:
statuses = [read_json(path).get("status") for path in PROGRESS_FILES.values()]
if not all(status in {"completed", "early_stopped"} for status in statuses):
    raise RuntimeError(f"아직 학습이 끝나지 않았습니다: {statuses}")
subprocess.run([str(PYTHON), "scripts/validation/evaluate_h_t05_h_t08_neighborhood.py", "--data-root", str(DATA_ROOT), "--radius", "4"], cwd=ROOT, check=True)


## 5. 최종 Macro Dice 요약


In [ ]:
rows = []
for name, progress_path in PROGRESS_FILES.items():
    fold = progress_path.parent
    best = read_json(fold / "best_full_macro.json")
    history = read_json(fold / "full_validation_history.json")
    rows.append({
        "실험": name,
        "최종 Best Macro Dice": best.get("macro_dice_positive"),
        "Best Epoch": best.get("epoch"),
        "양성 Case 수": best.get("positive_cases"),
        "평가한 Epoch 수": len(history) if isinstance(history, list) else 0,
        "보존 Checkpoints": len(list((fold / "epoch_checkpoints").glob("checkpoint_epoch_*.pth"))),
    })
leaderboard = pd.DataFrame(rows).sort_values("최종 Best Macro Dice", ascending=False)
display(leaderboard)
leaderboard.to_csv(BATCH_DIR / "full_macro_leaderboard.csv", index=False)
print("저장:", BATCH_DIR / "full_macro_leaderboard.csv")
